In [45]:
import webcam
import sys
import cv2
import time
from time import time
import datasetMng
import json
from mqtt_wrapper import MQTTWrapper

# Libraries
import numpy as np
from tqdm import tqdm
from PIL import Image

import torch
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, Conv2d, AvgPool2d, BatchNorm2d, Dropout2d, LeakyReLU, Linear, Module
from torch.utils.data import DataLoader

from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder

In [3]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
datasetMng.dataTransformer()

In [33]:
# Create Transformer Object

tfm_obj = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],[0.5, 0.5, 0.5])
])

In [35]:
# Build Model
class DistractionCNN(Module):
    def __init__(self):
        super(DistractionCNN, self).__init__()
        self.conv = Conv2d(in_channels=3, out_channels=8, kernel_size=(3,3), stride=1, padding=1)
        self.pool = AvgPool2d(kernel_size=(3,3), stride=1)
        self.relu = LeakyReLU()
        self.bn = BatchNorm2d(num_features=8)
        self.drop = Dropout2d(p=0.3)
        self.fc = Linear(in_features=8*126*126, out_features=5)
        
    def forward(self, X):
        output = self.conv(X)
        output = self.pool(output)
        output = self.relu(output)
        output = self.bn(output)
        output = self.drop(output)
#         print(output.shape)
        output = output.view(-1, 8*126*126)
        output = self.fc(output)
        return output

    def predict(image):
        display(image.resize((128, 128)))
        image_tensor = tfm_obj(image)
        image_tensor = image_tensor[np.newaxis, :]
        image_tensor = image_tensor.to(device)
        prob = model(image_tensor)
        pred = torch.max(prob, 1).indices
        pred = pred.item()
        if pred == 0:
            print(f"Model Pred: {pred}\nInterpretation: Other activities")
        elif pred == 1:
            print(f"Model Pred: {pred}\nInterpretation: Safe driving")
        elif pred == 2:
            print(f"Model Pred: {pred}\nInterpretation: Talking phone")
        elif pred == 3:
            print(f"Model Pred: {pred}\nInterpretation: Texting phone")
        else:
            print(f"Model Pred: {pred}\nInterpretation: Turning")
            
        print("===============================================================")

In [9]:
# Define Model and Migrate to GPU
model = DistractionCNN().to(device)
# Optimiser
optimiser = Adam(model.parameters(), lr=1e-4, weight_decay=0.0001)

# Loss Function
loss_fn = CrossEntropyLoss()

In [15]:
for epoch in range(25):
    start = time()
    train_acc = 0
    test_acc = 0

    # Train
    model.train()
    with tqdm(datasetMng.train_loader, unit="batch") as tepoch:
        for xtrain, ytrain in tepoch:
            optimiser.zero_grad()
            xtrain = xtrain.to(device)
            train_prob = model(xtrain)
            train_prob = train_prob.cpu()
            train_loss = loss_fn(train_prob, ytrain)
            train_loss.backward()
            optimiser.step()
            # END TRAIN
            train_pred = torch.max(train_prob, 1).indices
            train_acc += int(torch.sum(train_pred == ytrain))
            
        train_epoch_accuracy = train_acc / datasetMng.len_train

    # Evaluate
    model.eval()
    with torch.no_grad():
        for xtest, ytest in datasetMng.test_loader:
            xtest = xtest.to(device)
            test_prob = model(xtest)
            test_prob = test_prob.cpu()
            test_loss = loss_fn(test_prob, ytest)
            test_pred = torch.max(test_prob, 1).indices
            test_acc += int(torch.sum(test_pred == ytest))
        
        test_epoch_accuracy = test_acc / datasetMng.len_test
    
    end = time()
    
    diff = end - start
    
    print(f"Epoch: {epoch+1}, Time: {diff}\nTr_loss: {train_loss}, Test_loss: {test_loss}\n,Tr_acc:{train_epoch_accuracy}, Test_acc: {test_epoch_accuracy}")

100%|██████████| 505/505 [02:04<00:00,  4.07batch/s]


Epoch: 1, Time: 138.06367325782776
Tr_loss: 0.9620632529258728, Test_loss: 0.8620949387550354
,Tr_acc:0.5425742574257426, Test_acc: 0.6309412861136999


100%|██████████| 505/505 [02:01<00:00,  4.14batch/s]


Epoch: 2, Time: 135.6038041114807
Tr_loss: 0.912250816822052, Test_loss: 1.0089776515960693
,Tr_acc:0.6791419141914191, Test_acc: 0.744641192917055


100%|██████████| 505/505 [02:02<00:00,  4.14batch/s]


...

100%|██████████| 505/505 [02:01<00:00,  4.15batch/s]


Epoch: 24, Time: 135.3533182144165
Tr_loss: 0.2946924567222595, Test_loss: 0.03210880607366562
,Tr_acc:0.8866006600660066, Test_acc: 0.9231127679403541


100%|██████████| 505/505 [01:46<00:00,  4.75batch/s]


Epoch: 25, Time: 116.77405261993408
Tr_loss: 0.612321138381958, Test_loss: 0.04488985985517502
,Tr_acc:0.8868646864686469, Test_acc: 0.9389561975768872


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\lucrezia.rettori\\Desktop\\Distraction_detection'

In [17]:
torch.save(model.state_dict(), sys.path[0] + '/cnnmodel.h5')

In [23]:
model.load_state_dict(torch.load('cnnmodel.h5'))

<All keys matched successfully>

In [55]:
def loadUserConfig():
    with open(sys.path[0] + "/userConfig.json") as json_file:
        return json.load(json_file)

client = MQTTWrapper.fromJsonConfig(sys.path[0] + "/mqttConfig.json")
user = loadUserConfig()
webcam.setup()

while True:
    try:
        distractionsMap = webcam.getDistractionType(model)
        dataToTransmit = { 
            user["user"]: {
                "other_activities": distractionsMap["Other activities"],
                "safe_driving": distractionsMap["Safe driving"],
                "talking_phone": distractionsMap["Talking on the phone"],
                "texting_phone": distractionsMap["Texting on the phone"],
                "turning": distractionsMap["Turning"]
            } 
        }
        client.publish(json.dumps(dataToTransmit))
        print(dataToTransmit)
        time.sleep(1)
    except webcam.NoFaceDetectedException:
        print("No face found!")

TypeError: 'int' object is not subscriptable

In [53]:
# prevents openCL usage and unnecessary logging messages
cv2.ocl.setUseOpenCL(False)

# dictionary which assigns each type of distraction a label
type_of_distraction_dict = {0: "Other activities", 1: "Safe driving", 2: "Talking on the phone", 3: "Texting on the phone", 4: "Turning"}

# start the webcam feed
cap = cv2.VideoCapture(0)

# Find haar cascade to draw bounding box around face
ret, frame = cap.read()
if not ret:
    print(0)

facecasc = cv2.CascadeClassifier(sys.path[0] + '/haarcascade_frontalface_default.xml')
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
faces = facecasc.detectMultiScale(gray,scaleFactor=1.3, minNeighbors=5)

prediction = 0
for (x, y, w, h) in faces:
    cv2.rectangle(frame, (x, y-50), (x+w, y+h+10), (255, 0, 0), 2)
    roi_gray = gray[y:y + h, x:x + w]
    cropped_img = np.expand_dims(np.expand_dims(cv2.resize(roi_gray, (48, 48)), -1), 0)
    prediction = model.predict(cropped_img)

data = {}
if type(prediction) is np.ndarray:
    for distraction_type in type_of_distraction_dict:
        data[type_of_distraction_dict[distraction_type]] = round(prediction[0][distraction_type].item(), 5)
else:
    raise NoFaceDetectedException

print(data)


0


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
